In [ ]:
!pip install -q xgboost imbalanced-learn scipy scikit-learn pandas numpy

# Eksperimen lintas dataset untuk prediksi defect
Notebook ini mempertahankan pipeline M2, M4, dan M6, lalu mengevaluasi seluruh dataset dengan repeated stratified cross-validation. Preprocessing dan undersampling hanya dipelajari dari training fold.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff
from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, average_precision_score, roc_auc_score)
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42
DATA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/DatasetRM')
DATA_PATHS = {
    # NASA MDP (ARFF)
    'PC1': DATA_ROOT / 'PC1.arff', 'PC2': DATA_ROOT / 'PC2.arff',
    'PC3': DATA_ROOT / 'PC3.arff', 'PC4': DATA_ROOT / 'PC4.arff',
    'PC5': DATA_ROOT / 'PC5.arff', 'KC1': DATA_ROOT / 'KC1.arff',
    'KC3': DATA_ROOT / 'KC3.arff', 'MC1': DATA_ROOT / 'MC1.arff',
    # PROMISE/JIRA-style CSV
    'ivy-1.4': DATA_ROOT / 'ivy-1.4.csv', 'ivy-2.0': DATA_ROOT / 'ivy-2.0.csv',
    'jedit-4.0': DATA_ROOT / 'jedit-4.0.csv', 'jedit-4.2': DATA_ROOT / 'jedit-4.2.csv',
    'Tomcat': DATA_ROOT / 'tomcat.csv', 'synapse-1.0': DATA_ROOT / 'synapse-1.0.csv',
    'xalan-2.4': DATA_ROOT / 'xalan-2.4.csv', 'xerces-1.2': DATA_ROOT / 'xerces-1.2.csv',
}
# NASA MDP memakai Defective Y/N. CSV memakai kolom target yang ditemukan
# dari kandidat di bawah; isikan nama kolom secara eksplisit di sini bila berbeda/ambigu.
TARGET_COLUMNS = {name: ('Defective' if name in {'PC1','PC2','PC3','PC4','PC5','KC1','KC3','MC1'} else None) for name in DATA_PATHS}
TARGET_CANDIDATES = ('defective','bug','bugs','buggy','is_buggy','is_defective','defects','label','class')
HMD_MAX_DEPTH = 3
HMD_MIN_MAJORITY_SIZE = 20
N_SPLITS = 5
N_REPEATS = 5
XGB_PARAMS = {'n_estimators':150,'max_depth':4,'learning_rate':0.05,
    'subsample':0.8,'colsample_bytree':0.8,'eval_metric':'logloss',
    'random_state':RANDOM_STATE,'n_jobs':-1}


Mounted at /content/drive


In [ ]:
import pandas as pd
from pathlib import Path
from scipy.io import arff

def quick_get_features(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError("File tidak ditemukan")
    if path.suffix.lower() == '.arff':
        raw, _ = arff.loadarff(path)
        frame = pd.DataFrame(raw)
    elif path.suffix.lower() == '.csv':
        frame = pd.read_csv(path)
    else:
        raise ValueError("Format tidak didukung")

    # Hapus baris duplikat seperti preprocessing utama
    frame = frame.drop_duplicates()

    # Jumlah fitur adalah total kolom dikurangi 1 kolom target
    features_count = frame.shape[1] - 1
    return features_count, frame.shape[0]

# Mencoba membaca dari file ekspor terlebih dahulu jika ada
file_path = Path('/content/drive/MyDrive/Colab Notebooks/DatasetRM/hasil_eksperimen/audit_dataset.csv')

if file_path.exists():
    audit_df_saved = pd.read_csv(file_path)
    feature_summary = audit_df_saved[['Dataset', 'Features', 'Rows', 'Status']].copy()
    display(feature_summary)
else:
    print("Membaca metadata dataset langsung dari Google Drive secara cepat...\n")
    quick_audit = []
    for name, path in DATA_PATHS.items():
        try:
            feats, rows = quick_get_features(path)
            quick_audit.append({
                'Dataset': name,
                'Features': feats,
                'Rows': rows,
                'Status': 'ready'
            })
        except Exception as e:
            quick_audit.append({
                'Dataset': name,
                'Features': None,
                'Rows': None,
                'Status': f'error: {str(e)}'
            })

    quick_df = pd.DataFrame(quick_audit)
    display(quick_df)

Membaca metadata dataset langsung dari Google Drive secara cepat...



,Dataset,Features,Rows,Status
0,PC1,37,711,ready
1,PC2,36,745,ready
2,PC3,37,1079,ready
3,PC4,37,1288,ready
4,PC5,38,1723,ready
5,KC1,21,1203,ready
6,KC3,39,194,ready
7,MC1,38,1998,ready
8,ivy-1.4,23,241,ready
9,ivy-2.0,23,352,ready


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.io import arff

def local_load_dataset(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Dataset tidak ditemukan: {path}')
    if path.suffix.lower() == '.arff':
        raw, _ = arff.loadarff(path)
        frame = pd.DataFrame(raw)
        for col in frame.select_dtypes(include=['object']).columns:
            frame[col] = frame[col].map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v)
        return frame
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Ekstensi tidak didukung: {path.suffix}')

def local_resolve_target_column(frame, dataset_name):
    configured = TARGET_COLUMNS.get(dataset_name)
    if configured:
        return configured
    normalized = {str(c).strip().lower(): c for c in frame.columns}
    matches = [normalized[name] for name in TARGET_CANDIDATES if name in normalized]
    if len(matches) != 1:
        raise ValueError(f'Target tidak dapat ditentukan untuk {dataset_name}')
    return matches[0]

print("=== PRAPEMROSESAN & ANALISIS DATA YANG DIHAPUS ===\n")

for name, path in DATA_PATHS.items():
    try:
        # 1. Load dataset
        frame = local_load_dataset(path)
        initial_rows = len(frame)

        # Identifikasi data duplikat sebelum dihapus
        duplicate_mask = frame.duplicated(keep='first')
        duplicates_df = frame[duplicate_mask]
        duplicate_count = len(duplicates_df)

        # 2. Hapus data duplikat
        frame = frame.drop_duplicates().reset_index(drop=True)

        target_col = local_resolve_target_column(frame, name)
        X = frame.drop(columns=[target_col]).copy()
        y = frame[target_col]

        # 3. Membersihkan Noise dan Kesalahan Ketik (Typo) pada kolom kategorikal/string
        categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns
        for col in categorical_cols:
            X[col] = X[col].astype(str).str.strip().str.lower()
            X[col] = X[col].replace({'': np.nan, 'nan': np.nan, 'null': np.nan})

        # 4. Deteksi & Imputasi Missing Values sebelum menangani outlier
        numeric_cols = X.select_dtypes(include=['number']).columns
        total_missing = X.isnull().sum().sum()

        # Imputasi Median untuk numerik
        for col in numeric_cols:
            if X[col].isnull().any():
                median_val = X[col].median()
                X[col] = X[col].fillna(median_val)

        # Imputasi Most Frequent untuk kategorikal
        for col in categorical_cols:
            if X[col].isnull().any():
                most_freq_val = X[col].mode()[0] if not X[col].mode().empty else "unknown"
                X[col] = X[col].fillna(most_freq_val)

        # 5. Menangani Outlier (Pencilan) dengan Clipping
        outlier_clipped_count = 0
        for col in numeric_cols:
            lower_limit = X[col].quantile(0.01)
            upper_limit = X[col].quantile(0.99)
            outliers = ((X[col] < lower_limit) | (X[col] > upper_limit)).sum()
            outlier_clipped_count += outliers
            X[col] = np.clip(X[col], lower_limit, upper_limit)

        print(f"Dataset: {name}")
        print(f"  - Baris Duplikat Dihapus     : {duplicate_count} baris dari total {initial_rows}")
        if duplicate_count > 0:
            print("  - Sampel data duplikat yang dihapus (maksimal 3 baris):")
            display(duplicates_df.head(3))
        else:
            print("  - Tidak ada data duplikat yang terdeteksi.")
        print(f"  - Missing Values Ditangani   : {total_missing}")
        print(f"  - Nilai Outlier Di-clip (1-99%): {outlier_clipped_count}")
        print(f"  - Status                     : Sukses dibersihkan & siap")
        print("-" * 80)

    except Exception as e:
        print(f"Dataset {name} gagal diproses: {e}")
        print("-" * 80)

=== PRAPEMROSESAN & ANALISIS DATA YANG DIHAPUS ===

Dataset: PC1
  - Baris Duplikat Dihapus     : 48 baris dari total 759
  - Sampel data duplikat yang dihapus (maksimal 3 baris):


,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
40,4.0,15.0,1.0,1.0,1.0,26.0,8.0,0.38,12.0,2.17,...,26.0,0.30,33.0,50.0,12.0,19.0,27.0,9.09,21.0,N
65,7.0,3.0,1.0,0.0,0.0,4.0,2.0,0.18,2.0,2.00,...,6.0,0.11,14.0,21.0,12.0,12.0,19.0,0.00,11.0,N
72,10.0,5.0,2.0,0.0,0.0,8.0,3.0,0.18,4.0,2.00,...,9.0,0.11,30.0,43.0,19.0,16.0,28.0,0.00,17.0,N


  - Missing Values Ditangani   : 0
  - Nilai Outlier Di-clip (1-99%): 364
  - Status                     : Sukses dibersihkan & siap
--------------------------------------------------------------------------------
Dataset: PC2
  - Baris Duplikat Dihapus     : 840 baris dari total 1585
  - Sampel data duplikat yang dihapus (maksimal 3 baris):


,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,DESIGN_COMPLEXITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
8,3.0,0.0,0.0,0.0,4.0,2.0,1.00,2.0,2.0,1.0,...,6.0,0.40,4.0,8.0,3.0,5.0,5.0,0.00,2.0,N
9,3.0,0.0,0.0,0.0,4.0,2.0,1.00,2.0,2.0,1.0,...,6.0,0.40,4.0,8.0,3.0,5.0,5.0,0.00,2.0,N
83,5.0,3.0,18.0,1.0,8.0,3.0,0.14,4.0,2.0,2.0,...,12.0,0.12,30.0,36.0,13.0,14.0,26.0,86.36,21.0,N


  - Missing Values Ditangani   : 0
  - Nilai Outlier Di-clip (1-99%): 354
  - Status                     : Sukses dibersihkan & siap
--------------------------------------------------------------------------------
Dataset: PC3
  - Baris Duplikat Dihapus     : 46 baris dari total 1125
  - Sampel data duplikat yang dihapus (maksimal 3 baris):


,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
16,2.0,3.0,2.0,0.0,0.0,4.0,2.0,0.25,2.0,2.0,...,7.0,0.18,18.0,22.0,9.0,9.0,11.0,0.0,8.0,N
17,2.0,3.0,2.0,0.0,0.0,4.0,2.0,0.25,2.0,2.0,...,7.0,0.18,18.0,22.0,9.0,9.0,11.0,0.0,8.0,N
18,2.0,3.0,2.0,0.0,0.0,4.0,2.0,0.25,2.0,2.0,...,7.0,0.18,18.0,22.0,9.0,9.0,11.0,0.0,8.0,N


... (7 outputs left out)

,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
36,0.0,3.0,4.0,0.0,0.0,4.0,2.0,0.25,2.0,2.0,...,9.0,0.22,16.0,30.0,9.0,11.0,9.0,0.0,8.0,N
81,0.0,7.0,5.0,0.0,0.0,10.0,4.0,0.29,4.0,2.5,...,17.0,0.27,23.0,42.0,15.0,13.0,15.0,0.0,14.0,N
133,1.0,3.0,3.0,0.0,0.0,4.0,2.0,0.25,2.0,2.0,...,8.0,0.20,14.0,23.0,8.0,12.0,10.0,0.0,8.0,N


  - Missing Values Ditangani   : 0
  - Nilai Outlier Di-clip (1-99%): 88
  - Status                     : Sukses dibersihkan & siap
--------------------------------------------------------------------------------
Dataset: MC1
  - Baris Duplikat Dihapus     : 7279 baris dari total 9277
  - Sampel data duplikat yang dihapus (maksimal 3 baris):


,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DESIGN_COMPLEXITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,N
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,2.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,N


  - Missing Values Ditangani   : 0
  - Nilai Outlier Di-clip (1-99%): 670
  - Status                     : Sukses dibersihkan & siap
--------------------------------------------------------------------------------
Dataset: ivy-1.4
  - Baris Duplikat Dihapus     : 0 baris dari total 241
  - Tidak ada data duplikat yang terdeteksi.
  - Missing Values Ditangani   : 0
  - Nilai Outlier Di-clip (1-99%): 50
  - Status                     : Sukses dibersihkan & siap
--------------------------------------------------------------------------------
Dataset: ivy-2.0
  - Baris Duplikat Dihapus     : 0 baris dari total 352
  - Tidak ada data duplikat yang terdeteksi.
  - Missing Values Ditangani   : 0
  - Nilai Outlier Di-clip (1-99%): 65
  - Status                     : Sukses dibersihkan & siap
--------------------------------------------------------------------------------
Dataset: jedit-4.0
  - Baris Duplikat Dihapus     : 0 baris dari total 306
  - Tidak ada data duplikat yang terdeteksi.
  - 

## Loader dan pemetaan target

In [26]:
def load_dataset(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Dataset tidak ditemukan: {path}')
    if path.suffix.lower() == '.arff':
        raw, _ = arff.loadarff(path)
        frame = pd.DataFrame(raw)
        for col in frame.select_dtypes(include=['object']).columns:
            frame[col] = frame[col].map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v)
        return frame
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Ekstensi tidak didukung: {path.suffix}')

def resolve_target_column(frame, dataset_name):
    configured = TARGET_COLUMNS.get(dataset_name)
    if configured:
        if configured not in frame.columns:
            raise ValueError(f'{dataset_name}: target {configured!r} tidak ada. Kolom: {list(frame.columns)}')
        return configured
    normalized = {str(c).strip().lower(): c for c in frame.columns}
    matches = [normalized[name] for name in TARGET_CANDIDATES if name in normalized]
    if len(matches) != 1:
        raise ValueError(f'{dataset_name}: target tidak dapat ditentukan secara unik (kandidat cocok: {matches}). Isi TARGET_COLUMNS[{dataset_name!r}] dengan nama kolom yang benar.')
    return matches[0]

def map_defect_target(series, dataset_name):
    values = series.dropna()
    if values.empty:
        raise ValueError(f'{dataset_name}: target seluruhnya kosong')
    normalized = values.astype(str).str.strip().str.lower()
    positive = {'y','yes','true','1','bug','buggy','defect','defective','positive'}
    negative = {'n','no','false','0','clean','non-defect','non_defect','nonbuggy','negative'}
    observed = set(normalized.unique())
    if observed <= positive | negative and observed & positive and observed & negative:
        mapped = series.astype(str).str.strip().str.lower().map(lambda v: 1 if v in positive else (0 if v in negative else np.nan))
        return mapped.astype('Int64').astype(int), sorted(observed)
    numeric = pd.to_numeric(series, errors='coerce')
    unique = set(numeric.dropna().unique())
    if unique == {0, 1}:
        return numeric.astype('Int64').astype(int), sorted(unique)
    if numeric.notna().all() and len(unique) >= 2 and min(unique) >= 0:
        return (numeric > 0).astype(int), sorted(unique)
    raise ValueError(f'{dataset_name}: label target tidak dikenal/ambigu: {sorted(observed)}.')

def prepare_dataset(name, path):
    # 1. Load dataset
    frame = load_dataset(path)

    # 2. Hapus duplikat secara konsisten
    frame = frame.drop_duplicates().reset_index(drop=True)

    target_col = resolve_target_column(frame, name)
    keep = frame[target_col].notna()
    frame = frame.loc[keep].reset_index(drop=True)

    # Pisahkan target dan fitur
    y, source_labels = map_defect_target(frame[target_col], name)
    X = frame.drop(columns=[target_col]).copy()

    # 3. Bersihkan Noise dan Typo pada kolom kategorikal/string
    categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns
    for col in categorical_cols:
        X[col] = X[col].astype(str).str.strip().str.lower()
        X[col] = X[col].replace({'': np.nan, 'nan': np.nan, 'null': np.nan})

    # 4. Deteksi & Imputasi Missing Values sebelum outlier clipping
    numeric_cols = X.select_dtypes(include=['number']).columns

    # Imputasi median untuk fitur numerik
    for col in numeric_cols:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())

    # Imputasi modus untuk fitur kategorikal
    for col in categorical_cols:
        if X[col].isnull().any():
            most_freq_val = X[col].mode()[0] if not X[col].mode().empty else "unknown"
            X[col] = X[col].fillna(most_freq_val)

    # 5. Menangani Outlier (Pencilan) dengan clipping (Persentil 1% - 99%)
    for col in numeric_cols:
        lower_limit = X[col].quantile(0.01)
        upper_limit = X[col].quantile(0.99)
        X[col] = np.clip(X[col], lower_limit, upper_limit)

    counts = y.value_counts().to_dict()
    if set(counts) != {0,1}:
        raise ValueError(f'{name}: setelah pemetaan target tidak ada dua kelas: {counts}')

    return X, y, {'Dataset':name,'Path':str(path),'Rows':len(frame),'Features':X.shape[1],
        'Target column':target_col,'Source labels':str(source_labels),'Non-defect (0)':counts[0],
        'Defect (1)':counts[1],'Status':'ready','Issue':''}

## Model dan evaluasi

In [27]:
def to_dense_array(x):
    return np.asarray(x.toarray() if hasattr(x, 'toarray') else x, dtype=np.float32)

def create_preprocessor(X_fit):
    numeric = X_fit.select_dtypes(include=['number']).columns.tolist()
    categorical = X_fit.select_dtypes(include=['object','category','bool']).columns.tolist()
    transformers = []
    if numeric:
        transformers.append(('numeric', Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]), numeric))
    if categorical:
        transformers.append(('categorical', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), categorical))
    if not transformers:
        raise ValueError('Tidak ada fitur numerik/kategorikal yang dapat diproses.')
    return ColumnTransformer(transformers=transformers, remainder='drop')

def create_xgb_binary(random_state):
    params = XGB_PARAMS.copy(); params.update(objective='binary:logistic', eval_metric='logloss', random_state=random_state)
    return XGBClassifier(**params)

def calculate_gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    sensitivity = tp/(tp+fn) if tp+fn else 0.0
    specificity = tn/(tn+fp) if tn+fp else 0.0
    return float(np.sqrt(sensitivity*specificity))

def evaluate_model(y_true, y_pred, model_name, y_prob=None):
    result = {'Model':model_name,'Precision':precision_score(y_true,y_pred,pos_label=1,zero_division=0),
      'Recall':recall_score(y_true,y_pred,pos_label=1,zero_division=0),'F1-score':f1_score(y_true,y_pred,pos_label=1,zero_division=0),
      'G-Mean':calculate_gmean(y_true,y_pred),'Balanced Accuracy':balanced_accuracy_score(y_true,y_pred),
      'MCC':matthews_corrcoef(y_true,y_pred)}
    if y_prob is not None:
        result['AUC-ROC']=roc_auc_score(y_true,y_prob)
        result['AUC-PR']=average_precision_score(y_true,y_prob)
    return result


## HMD + local XGBoost

In [28]:
class HMDXGBoost:
    """
    Hierarchical Majority Decomposition + XGBoost.

    Target global:
        0 = non-defect / majority
        1 = defect / minority

    Internal node:
        0 = defect
        1 = majority child 1
        2 = majority child 2

    Leaf:
        0 = non-defect
        1 = defect
    """

    def __init__(
        self,
        max_depth=3,
        min_majority_size=20,
        random_state=42,
        xgb_params=None
    ):
        self.max_depth = max_depth
        self.min_majority_size = min_majority_size
        self.random_state = random_state

        self.xgb_params = xgb_params or XGB_PARAMS.copy()

        self.root = None
        self.X_train = None
        self.y_train = None
        self.minority_indices = None

    def _make_binary_model(self, random_state):
        params = self.xgb_params.copy()

        params.update({
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": random_state
        })

        params.pop("num_class", None)

        return XGBClassifier(**params)

    def _make_multiclass_model(self, random_state):
        params = self.xgb_params.copy()

        params.update({
            "objective": "multi:softprob",
            "num_class": 3,
            "eval_metric": "mlogloss",
            "random_state": random_state
        })

        return XGBClassifier(**params)

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y, dtype=int)

        self.minority_indices = np.where(
            self.y_train == 1
        )[0]

        majority_indices = np.where(
            self.y_train == 0
        )[0]

        if len(self.minority_indices) == 0:
            raise ValueError(
                "Tidak ditemukan sampel defect pada training data."
            )

        if len(majority_indices) < 2:
            raise ValueError(
                "Jumlah sampel mayoritas tidak cukup untuk K-Means."
            )

        self.root = self._build_node(
            majority_indices=majority_indices,
            depth=0
        )

        return self

    def _build_node(self, majority_indices, depth):
        node = {
            "depth": depth,
            "majority_indices": np.asarray(majority_indices),
            "is_leaf": False,
            "model": None,
            "left": None,
            "right": None,
            "kmeans": None
        }

        # Kondisi berhenti: node menjadi leaf
        stop_decomposition = (
            depth >= self.max_depth
            or len(majority_indices) < self.min_majority_size
            or len(majority_indices) < 2
        )

        if stop_decomposition:
            node["is_leaf"] = True

            # Data leaf:
            # minority defect + majority yang berada di leaf.
            leaf_indices = np.concatenate([
                self.minority_indices,
                majority_indices
            ])

            X_leaf = self.X_train[leaf_indices]
            y_leaf = self.y_train[leaf_indices]

            # Classifier leaf memprediksi:
            # 0 = non-defect
            # 1 = defect
            leaf_model = self._make_binary_model(
                self.random_state + depth
            )

            leaf_model.fit(X_leaf, y_leaf)

            node["model"] = leaf_model

            return node

        # K-Means hanya diterapkan pada kelas mayoritas
        X_majority = self.X_train[majority_indices]

        kmeans = KMeans(
            n_clusters=2,
            random_state=self.random_state + depth,
            n_init=10
        )

        cluster_labels = kmeans.fit_predict(X_majority)

        cluster_0_indices = majority_indices[
            cluster_labels == 0
        ]

        cluster_1_indices = majority_indices[
            cluster_labels == 1
        ]

        # Jika salah satu cluster kosong, gunakan leaf
        if (
            len(cluster_0_indices) == 0
            or len(cluster_1_indices) == 0
        ):
            node["is_leaf"] = True

            leaf_indices = np.concatenate([
                self.minority_indices,
                majority_indices
            ])

            X_leaf = self.X_train[leaf_indices]
            y_leaf = self.y_train[leaf_indices]

            leaf_model = self._make_binary_model(
                self.random_state + depth
            )

            leaf_model.fit(X_leaf, y_leaf)

            node["model"] = leaf_model

            return node

        node["kmeans"] = kmeans

        # Dataset lokal internal node:
        # seluruh minority + majority child 1 + majority child 2.
        local_indices = np.concatenate([
            self.minority_indices,
            cluster_0_indices,
            cluster_1_indices
        ])

        X_local = self.X_train[local_indices]

        y_local = np.concatenate([
            np.zeros(
                len(self.minority_indices),
                dtype=int
            ),
            np.ones(
                len(cluster_0_indices),
                dtype=int
            ),
            np.full(
                len(cluster_1_indices),
                2,
                dtype=int
            )
        ])

        local_model = self._make_multiclass_model(
            self.random_state + depth
        )

        local_model.fit(
            X_local,
            y_local
        )

        node["model"] = local_model

        # Recursive decomposition pada kelas mayoritas
        node["left"] = self._build_node(
            majority_indices=cluster_0_indices,
            depth=depth + 1
        )

        node["right"] = self._build_node(
            majority_indices=cluster_1_indices,
            depth=depth + 1
        )

        return node

    def _predict_one(self, x, node):
        # Leaf: klasifikasi defect vs non-defect
        if node["is_leaf"]:
            pred = int(
                node["model"].predict(
                    x.reshape(1, -1)
                )[0]
            )

            return pred

        # Internal node: prediksi kelas lokal
        local_pred = int(
            node["model"].predict(
                x.reshape(1, -1)
            )[0]
        )

        # Local label 0 = defect
        if local_pred == 0:
            return 1

        # Local label 1 = majority child 1
        if local_pred == 1:
            return self._predict_one(
                x,
                node["left"]
            )

        # Local label 2 = majority child 2
        if local_pred == 2:
            return self._predict_one(
                x,
                node["right"]
            )

        # Fallback defensif
        return 0

    def predict(self, X):
        X = np.asarray(X)

        predictions = []

        for x in X:
            predictions.append(
                self._predict_one(x, self.root)
            )

        return np.asarray(
            predictions,
            dtype=int
        )

    def predict_proba(self, X):
        """
        Menghasilkan probabilitas defect dari model leaf
        yang dicapai oleh masing-masing sampel.

        Ini adalah probabilitas lokal pada leaf, bukan
        probabilitas global yang dikalibrasi sepanjang jalur.
        """
        X = np.asarray(X)

        probabilities = []

        for x in X:
            node = self.root

            while not node["is_leaf"]:
                local_pred = int(
                    node["model"].predict(
                        x.reshape(1, -1)
                    )[0]
                )

                if local_pred == 0:
                    # Internal node memprediksi defect.
                    # Nilai ini merupakan confidence lokal
                    # dari kelas defect pada node.
                    local_prob = node["model"].predict_proba(
                        x.reshape(1, -1)
                    )[0][0]

                    probabilities.append(
                        float(local_prob)
                    )

                    break

                elif local_pred == 1:
                    node = node["left"]

                else:
                    node = node["right"]

            else:
                # Sampel mencapai leaf
                prob = node["model"].predict_proba(
                    x.reshape(1, -1)
                )[0][1]

                probabilities.append(
                    float(prob)
                )

        return np.asarray(probabilities)

## Jalankan semua dataset: preprocessing, RUS, dan HMD di dalam setiap fold

In [29]:
cv_results = []
audit_rows = []
model_names = ['M2 - XGBoost','M4 - XGBoost + Random Undersampling','M6 - HMD + XGBoost lokal']

for dataset_number, (dataset_name, dataset_path) in enumerate(DATA_PATHS.items(), start=1):
    print(f'\n[{dataset_number}/{len(DATA_PATHS)}] {dataset_name}: {dataset_path}')
    try:
        X, y, audit = prepare_dataset(dataset_name, dataset_path)
        if y.value_counts().min() < N_SPLITS:
            raise ValueError(f'Sampel kelas minoritas ({y.value_counts().min()}) kurang dari N_SPLITS={N_SPLITS}')
        audit_rows.append(audit)
        dataset_cv = RepeatedStratifiedKFold(n_splits=N_SPLITS,n_repeats=N_REPEATS,random_state=RANDOM_STATE)
        for fold_number, (train_idx, valid_idx) in enumerate(dataset_cv.split(X,y), start=1):
            X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx].to_numpy(dtype=int), y.iloc[valid_idx].to_numpy(dtype=int)
            seed = RANDOM_STATE + dataset_number*1000 + fold_number
            # Imputer, scaler, dan encoder di-fit hanya pada training fold.
            preprocessor = create_preprocessor(X_train)
            X_train_p = to_dense_array(preprocessor.fit_transform(X_train))
            X_valid_p = to_dense_array(preprocessor.transform(X_valid))

            m2 = create_xgb_binary(seed).fit(X_train_p,y_train)
            pred_m2 = m2.predict(X_valid_p); prob_m2 = m2.predict_proba(X_valid_p)[:,1]

            # RUS melihat training fold saja; validation fold tetap pada distribusi asli.
            X_rus,y_rus = RandomUnderSampler(random_state=seed).fit_resample(X_train_p,y_train)
            m4 = create_xgb_binary(seed).fit(X_rus,y_rus)
            pred_m4 = m4.predict(X_valid_p); prob_m4 = m4.predict_proba(X_valid_p)[:,1]

            # Objek HMD baru diinisialisasi dan di-fit ulang untuk setiap fold.
            m6 = HMDXGBoost(max_depth=HMD_MAX_DEPTH,min_majority_size=HMD_MIN_MAJORITY_SIZE,random_state=seed,xgb_params=XGB_PARAMS)
            m6.fit(X_train_p,y_train)
            pred_m6 = m6.predict(X_valid_p)

            fold_outputs = [(model_names[0],pred_m2,prob_m2),(model_names[1],pred_m4,prob_m4),(model_names[2],pred_m6,None)]
            for model_name,pred,prob in fold_outputs:
                row = evaluate_model(y_valid,pred,model_name,y_prob=prob)
                row.update({'Dataset':dataset_name,'Fold':fold_number})
                cv_results.append(row)
        print(f'  selesai: {N_SPLITS*N_REPEATS} folds')
    except Exception as exc:
        # Catat masalah per dataset agar dataset lain tetap dapat diproses.
        audit_rows.append({'Dataset':dataset_name,'Path':str(dataset_path),'Rows':None,'Features':None,
            'Target column':TARGET_COLUMNS.get(dataset_name),'Source labels':None,'Non-defect (0)':None,'Defect (1)':None,
            'Status':'error','Issue':str(exc)})
        print(f'  dilewati: {exc}')

cv_results_df = pd.DataFrame(cv_results)
audit_df = pd.DataFrame(audit_rows)
if cv_results_df.empty:
    raise RuntimeError('Tidak ada dataset yang berhasil dievaluasi. Periksa audit_dataset.csv dan pemetaan target.')
display(audit_df)
display(cv_results_df.head())



[1/16] PC1: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC1.arff
  selesai: 25 folds

[2/16] PC2: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC2.arff
  selesai: 25 folds

[3/16] PC3: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC3.arff
  selesai: 25 folds

[4/16] PC4: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC4.arff
  selesai: 25 folds

[5/16] PC5: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC5.arff
  selesai: 25 folds

[6/16] KC1: /content/drive/MyDrive/Colab Notebooks/DatasetRM/KC1.arff
  selesai: 25 folds

[7/16] KC3: /content/drive/MyDrive/Colab Notebooks/DatasetRM/KC3.arff
  selesai: 25 folds

[8/16] MC1: /content/drive/MyDrive/Colab Notebooks/DatasetRM/MC1.arff
  selesai: 25 folds

[9/16] ivy-1.4: /content/drive/MyDrive/Colab Notebooks/DatasetRM/ivy-1.4.csv
  selesai: 25 folds

[10/16] ivy-2.0: /content/drive/MyDrive/Colab Notebooks/DatasetRM/ivy-2.0.csv
  selesai: 25 folds

[11/16] jedit-4.0: /content/drive/MyDrive/Colab Notebooks/DatasetRM/jedit-

,Dataset,Path,Rows,Features,Target column,Source labels,Non-defect (0),Defect (1),Status,Issue
0,PC1,/content/drive/MyDrive/Colab Notebooks/Dataset...,711,37,Defective,"['n', 'y']",650,61,ready,
1,PC2,/content/drive/MyDrive/Colab Notebooks/Dataset...,745,36,Defective,"['n', 'y']",729,16,ready,
2,PC3,/content/drive/MyDrive/Colab Notebooks/Dataset...,1079,37,Defective,"['n', 'y']",945,134,ready,
3,PC4,/content/drive/MyDrive/Colab Notebooks/Dataset...,1288,37,Defective,"['n', 'y']",1111,177,ready,
4,PC5,/content/drive/MyDrive/Colab Notebooks/Dataset...,1723,38,Defective,"['n', 'y']",1252,471,ready,
5,KC1,/content/drive/MyDrive/Colab Notebooks/Dataset...,1203,21,Defective,"['n', 'y']",889,314,ready,
6,KC3,/content/drive/MyDrive/Colab Notebooks/Dataset...,194,39,Defective,"['n', 'y']",158,36,ready,
7,MC1,/content/drive/MyDrive/Colab Notebooks/Dataset...,1998,38,Defective,"['n', 'y']",1952,46,ready,
8,ivy-1.4,/content/drive/MyDrive/Colab Notebooks/Dataset...,241,23,bug,"[np.int64(0), np.int64(1), np.int64(3)]",225,16,ready,
9,ivy-2.0,/content/drive/MyDrive/Colab Notebooks/Dataset...,352,23,bug,"[np.int64(0), np.int64(1), np.int64(2), np.int...",312,40,ready,


,Model,Precision,Recall,F1-score,G-Mean,Balanced Accuracy,MCC,AUC-ROC,AUC-PR,Dataset,Fold
0,M2 - XGBoost,0.500000,0.153846,0.235294,0.389203,0.569231,0.241399,0.858284,0.422281,PC1,1
1,M4 - XGBoost + Random Undersampling,0.204545,0.692308,0.315789,0.711279,0.711538,0.263523,0.857396,0.557183,PC1,1
2,M6 - HMD + XGBoost lokal,0.538462,0.538462,0.538462,0.716666,0.746154,0.492308,NaN,NaN,PC1,1
3,M2 - XGBoost,0.400000,0.166667,0.235294,0.403510,0.571795,0.216691,0.833013,0.339232,PC1,2
4,M4 - XGBoost + Random Undersampling,0.177778,0.666667,0.280702,0.690596,0.691026,0.228398,0.795833,0.202409,PC1,2


## Ringkasan per dataset dan model

In [30]:
metric_columns = ['Precision','Recall','F1-score','G-Mean','Balanced Accuracy','MCC']
if 'AUC-ROC' in cv_results_df.columns:
    metric_columns.append('AUC-ROC')
if 'AUC-PR' in cv_results_df.columns:
    metric_columns.append('AUC-PR')

# Menghitung ringkasan per dataset dan model berdasarkan hasil CV
summary_by_dataset = cv_results_df.groupby(['Dataset','Model'])[metric_columns].agg(['mean','std'])
summary_overall = cv_results_df.groupby('Model')[metric_columns].agg(['mean','std'])

print("=== RINGKASAN EVALUASI PER DATASET (RATA-RATA & STANDAR DEVIASI) ===")
display(summary_by_dataset)

print("\n=== RINGKASAN EVALUASI KESELURUHAN ===")
display(summary_overall)

=== RINGKASAN EVALUASI PER DATASET (RATA-RATA & STANDAR DEVIASI) ===


Precision              Recall  \
                                                     mean       std      mean   
Dataset     Model                                                               
KC1         M2 - XGBoost                         0.621894  0.073550  0.254859   
            M4 - XGBoost + Random Undersampling  0.375103  0.025874  0.583564   
            M6 - HMD + XGBoost lokal             0.483418  0.044167  0.419857   
KC3         M2 - XGBoost                         0.406762  0.323284  0.182143   
            M4 - XGBoost + Random Undersampling  0.290570  0.055934  0.623571   
            M6 - HMD + XGBoost lokal             0.356901  0.116815  0.466429   
MC1         M2 - XGBoost                         0.136667  0.260075  0.033778   
            M4 - XGBoost + Random Undersampling  0.062733  0.009955  0.796000   
            M6 - HMD + XGBoost lokal             0.251290  0.205367  0.137333   
PC1         M2 - XGBoost                         0.445604  0.218486  0.176923   
            M4 - XGBoost + Random Undersampling  0.212574  0.036417  0.781026   
            M6 - HMD + XGBoost lokal             0.332315  0.129558  0.361795   
PC2         M2 - XGBoost                         0.000000  0.000000  0.000000   
            M4 - XGBoost + Random Undersampling  0.055873  0.015943  0.716667   
            M6 - HMD + XGBoost lokal             0.050000  0.117851  0.050000   
PC3         M2 - XGBoost                         0.446308  0.134013  0.167179   
            M4 - XGBoost + Random Undersampling  0.275309  0.020715  0.783761   
            M6 - HMD + XGBoost lokal             0.346907  0.079198  0.373618   
PC4         M2 - XGBoost                         0.682016  0.076830  0.482635   
            M4 - XGBoost + Random Undersampling  0.423451  0.037006  0.898540   
            M6 - HMD + XGBoost lokal             0.537098  0.051914  0.653492   
PC5         M2 - XGBoost                         0.627541  0.065937  0.381725   
            M4 - XGBoost + Random Undersampling  0.459533  0.026295  0.722244   
            M6 - HMD + XGBoost lokal             0.537258  0.043404  0.519709   
Tomcat      M2 - XGBoost                         0.471837  0.182622  0.184500   
            M4 - XGBoost + Random Undersampling  0.233990  0.030588  0.743667   
            M6 - HMD + XGBoost lokal             0.301368  0.085883  0.332000   
ivy-1.4     M2 - XGBoost                         0.000000  0.000000  0.000000   
            M4 - XGBoost + Random Undersampling  0.103099  0.052086  0.620000   
            M6 - HMD + XGBoost lokal             0.059873  0.089313  0.123333   
ivy-2.0     M2 - XGBoost                         0.420048  0.223014  0.210000   
            M4 - XGBoost + Random Undersampling  0.240822  0.047379  0.730000   
            M6 - HMD + XGBoost lokal             0.329372  0.108235  0.440000   
jedit-4.0   M2 - XGBoost                         0.705270  0.135934  0.501333   
            M4 - XGBoost + Random Undersampling  0.485544  0.079884  0.712000   
            M6 - HMD + XGBoost lokal             0.524820  0.088686  0.632000   
jedit-4.2   M2 - XGBoost                         0.567016  0.235932  0.270222   
            M4 - XGBoost + Random Undersampling  0.318381  0.052602  0.761778   
            M6 - HMD + XGBoost lokal             0.395273  0.087161  0.524444   
synapse-1.0 M2 - XGBoost                         0.142667  0.260078  0.103333   
            M4 - XGBoost + Random Undersampling  0.240644  0.094343  0.723333   
            M6 - HMD + XGBoost lokal             0.142765  0.162751  0.243333   
xalan-2.4   M2 - XGBoost                         0.536066  0.177464  0.243636   
            M4 - XGBoost + Random Undersampling  0.296323  0.032651  0.729091   
            M6 - HMD + XGBoost lokal             0.372594  0.074803  0.503636   
xerces-1.2  M2 - XGBoost                         0.532105  0.176654  0.251048   
            M4 - XGBoost + Random Undersampling  0.304004  0.054108  0.684381  


=== RINGKASAN EVALUASI KESELURUHAN ===


Precision              Recall            \
                                         mean       std      mean       std   
Model                                                                         
M2 - XGBoost                         0.421363  0.287056  0.215207  0.173030   
M4 - XGBoost + Random Undersampling  0.273622  0.132181  0.725601  0.176264   
M6 - HMD + XGBoost lokal             0.336295  0.181358  0.386811  0.215179   

                                     F1-score              G-Mean            \
                                         mean       std      mean       std   
Model                                                                         
M2 - XGBoost                         0.274281  0.200359  0.381577  0.242569   
M4 - XGBoost + Random Undersampling  0.377401  0.153278  0.708143  0.110447   
M6 - HMD + XGBoost lokal             0.350828  0.186556  0.527422  0.236206   

                                    Balanced Accuracy                 MCC  \
                                                 mean       std      mean   
Model                                                                       
M2 - XGBoost                                 0.588788  0.079607  0.233565   
M4 - XGBoost + Random Undersampling          0.717784  0.092427  0.294548   
M6 - HMD + XGBoost lokal                     0.634215  0.095997  0.254195   

                                                AUC-ROC              AUC-PR  \
                                          std      mean       std      mean   
Model                                                                         
M2 - XGBoost                         0.190407  0.792177  0.102033  0.425767   
M4 - XGBoost + Random Undersampling  0.136597  0.778009  0.098808  0.377448   
M6 - HMD + XGBoost lokal             0.175710       NaN       NaN       NaN   

                                               
                                          std  
Model                                          
M2 - XGBoost                         0.172574  
M4 - XGBoost + Random Undersampling  0.169812  
M6 - HMD + XGBoost lokal                  NaN

## Uji Wilcoxon berpasangan per dataset

In [31]:
def paired_wilcoxon_by_dataset(cv_df, model_a, model_b, metric='F1-score'):
    rows=[]
    for dataset, group in cv_df.groupby('Dataset'):
        a=group[group.Model==model_a].sort_values('Fold')[metric].to_numpy()
        b=group[group.Model==model_b].sort_values('Fold')[metric].to_numpy()
        if len(a)!=len(b) or len(a)<2 or np.allclose(a,b):
            rows.append({'Dataset':dataset,'Model A':model_a,'Model B':model_b,'Metric':metric,'p-value':np.nan,'Note':'tidak cukup pasangan atau seluruh skor sama'})
            continue
        stat,p=wilcoxon(a,b,alternative='two-sided')
        rows.append({'Dataset':dataset,'Model A':model_a,'Model B':model_b,'Metric':metric,'Statistic':stat,'p-value':p,'Note':''})
    return pd.DataFrame(rows)

wilcoxon_m6_m2 = paired_wilcoxon_by_dataset(cv_results_df,model_names[0],model_names[2])
wilcoxon_m6_m4 = paired_wilcoxon_by_dataset(cv_results_df,model_names[1],model_names[2])
display(wilcoxon_m6_m2); display(wilcoxon_m6_m4)


,Dataset,Model A,Model B,Metric,Statistic,p-value,Note
0,KC1,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,0.0,5.960464e-08,
1,KC3,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,39.0,1.516067e-03,
2,MC1,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,3.0,3.259445e-04,
3,PC1,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,36.0,1.920069e-03,
4,PC2,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,0.0,6.331779e-02,
5,PC3,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,8.0,1.490116e-06,
6,PC4,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,64.0,6.725550e-03,
7,PC5,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,8.0,1.490116e-06,
8,Tomcat,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,72.0,1.487962e-02,
9,ivy-1.4,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,0.0,7.579282e-03,


,Dataset,Model A,Model B,Metric,Statistic,p-value,Note
0,KC1,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,135.5,0.467520,
1,KC3,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,141.0,0.797068,
2,MC1,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,88.0,0.045123,
3,PC1,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,138.0,0.524913,
4,PC2,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,94.0,0.065238,
5,PC3,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,65.0,0.007371,
6,PC4,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,107.0,0.140915,
7,PC5,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,19.0,0.000182,
8,Tomcat,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,74.0,0.015973,
9,ivy-1.4,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,33.0,0.002399,


## Audit dan ekspor hasil

In [32]:
OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/DatasetRM/hasil_eksperimen')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Simpan metrik hasil evaluasi
try:
    audit_df.to_csv(OUTPUT_DIR/'audit_dataset.csv', index=False)
    cv_results_df.to_csv(OUTPUT_DIR/'hasil_repeated_stratified_cv.csv', index=False)
    summary_by_dataset.to_csv(OUTPUT_DIR/'ringkasan_cv_per_dataset.csv')
    summary_overall.to_csv(OUTPUT_DIR/'ringkasan_repeated_stratified_cv.csv')
    wilcoxon_m6_m2.to_csv(OUTPUT_DIR/'wilcoxon_m6_vs_m2_per_dataset.csv', index=False)
    wilcoxon_m6_m4.to_csv(OUTPUT_DIR/'wilcoxon_m6_vs_m4_per_dataset.csv', index=False)
    print(f'Hasil evaluasi model tersimpan di {OUTPUT_DIR}')
except NameError as e:
    print(f'Catatan ekspor evaluasi ditunda karena model belum selesai dijalankan: {e}')

# 2. Simpan setiap dataset hasil prapemrosesan dari Cell 5 ke folder eksperimen
print("\n=== MENYIMPAN DATASET HASIL PRAPEMROSESAN (CELL 5) ===")
for name, path in DATA_PATHS.items():
    try:
        # Gunakan fungsi prapemrosesan terintegrasi dari Cell 7
        X_clean, y_clean, _ = prepare_dataset(name, path)
        # Gabungkan kembali fitur dengan target terpetakan
        clean_df = X_clean.copy()
        clean_df['target_class'] = y_clean

        # Simpan ke file csv
        clean_path = OUTPUT_DIR / f'{name}_clean.csv'
        clean_df.to_csv(clean_path, index=False)
        print(f'  - Dataset {name} (bersih) berhasil disimpan -> {clean_path.name}')
    except Exception as exc:
        print(f'  - Gagal menyimpan dataset {name}: {exc}')

Hasil evaluasi model tersimpan di /content/drive/MyDrive/Colab Notebooks/DatasetRM/hasil_eksperimen

=== MENYIMPAN DATASET HASIL PRAPEMROSESAN (CELL 5) ===
  - Dataset PC1 (bersih) berhasil disimpan -> PC1_clean.csv
  - Dataset PC2 (bersih) berhasil disimpan -> PC2_clean.csv
  - Dataset PC3 (bersih) berhasil disimpan -> PC3_clean.csv
  - Dataset PC4 (bersih) berhasil disimpan -> PC4_clean.csv
  - Dataset PC5 (bersih) berhasil disimpan -> PC5_clean.csv
  - Dataset KC1 (bersih) berhasil disimpan -> KC1_clean.csv
  - Dataset KC3 (bersih) berhasil disimpan -> KC3_clean.csv
  - Dataset MC1 (bersih) berhasil disimpan -> MC1_clean.csv
  - Dataset ivy-1.4 (bersih) berhasil disimpan -> ivy-1.4_clean.csv
  - Dataset ivy-2.0 (bersih) berhasil disimpan -> ivy-2.0_clean.csv
  - Dataset jedit-4.0 (bersih) berhasil disimpan -> jedit-4.0_clean.csv
  - Dataset jedit-4.2 (bersih) berhasil disimpan -> jedit-4.2_clean.csv
  - Dataset Tomcat (bersih) berhasil disimpan -> Tomcat_clean.csv
  - Dataset synaps

In [33]:
OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/DatasetRM/hasil_eksperimen_finalcopy')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
audit_df.to_csv(OUTPUT_DIR/'audit_dataset.csv',index=False)
cv_results_df.to_csv(OUTPUT_DIR/'hasil_repeated_stratified_cv.csv',index=False)
summary_by_dataset.to_csv(OUTPUT_DIR/'ringkasan_cv_per_dataset.csv')
summary_overall.to_csv(OUTPUT_DIR/'ringkasan_repeated_stratified_cv.csv')
wilcoxon_m6_m2.to_csv(OUTPUT_DIR/'wilcoxon_m6_vs_m2_per_dataset.csv',index=False)
wilcoxon_m6_m4.to_csv(OUTPUT_DIR/'wilcoxon_m6_vs_m4_per_dataset.csv',index=False)
print(f'Hasil tersimpan di {OUTPUT_DIR}')
print('Status audit:',audit_df['Status'].value_counts(dropna=False).to_dict())


Hasil tersimpan di /content/drive/MyDrive/Colab Notebooks/DatasetRM/hasil_eksperimen_finalcopy
Status audit: {'ready': 16}
